# 5 · Exporting the models for the browser

Free Hugging Face Spaces no longer run Gradio apps, so the slide deck runs the models itself, in the viewer's
browser, with onnxruntime-web. This notebook shows the export (`export_onnx.py` does it for all models) and the
check that the browser runs exactly what was trained.

One ONNX graph serves both frozen-backbone methods: it takes the image *and* a prompt tensor of shape
(B, 12, P, 768). With P = 0 it is the plain base model (the linear probe's features); with P = 50 it is
VPT-deep. A mixed batch can even give every image its own task's prompts. The fully fine-tuned models
are the same graph with their own weights.

In [1]:
import sys, json, time, math
from pathlib import Path
ROOT = Path.cwd().parent            # demo/vpt
sys.path.insert(0, str(ROOT))
import numpy as np, torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

torch 2.11.0+cu128 | CUDA: NVIDIA GeForce RTX 5070 Ti Laptop GPU


In [2]:
import torch.nn as nn, onnx, onnxruntime as ort
from onnxruntime.transformers.float16 import convert_float_to_float16
from vpt_model import VPTViT

class Encoder(nn.Module):
    """ViT-B/16 whose every layer takes a (possibly empty) set of prompts; returns the final [CLS] feature."""
    def __init__(self, vit):
        super().__init__(); self.vit = vit
    def forward(self, image, prompts):                # prompts: (B, 12, P, 768)
        v, p = self.vit, prompts.shape[2]
        x = v.norm_pre(v._pos_embed(v.patch_embed(image)))
        for i, blk in enumerate(v.blocks):
            rest = x[:, 1:] if i == 0 else x[:, 1 + p:]
            x = blk(torch.cat([x[:, :1], prompts[:, i], rest], dim=1))
        return v.norm(x)[:, 0]

torch.set_grad_enabled(False)
enc = Encoder(VPTViT(2, num_prompts=0).vit.eval()).eval()
tmp = ROOT / "web" / "notebook_check"; tmp.mkdir(parents=True, exist_ok=True)
fp32, fp16 = tmp / "encoder.fp32.onnx", tmp / "encoder.fp16.onnx"
torch.onnx.export(enc, (torch.randn(2, 3, 224, 224), torch.randn(2, 12, 50, 768)), fp32, input_names=["image", "prompts"], output_names=["cls"],
                  opset_version=17, dynamic_axes={"image": {0: "B"}, "prompts": {0: "B", 2: "P"}, "cls": {0: "B"}}, dynamo=False)
onnx.save(convert_float_to_float16(onnx.load(str(fp32)), keep_io_types=True), str(fp16))
print(f"fp32 {fp32.stat().st_size / 1e6:.0f} MB -> fp16 {fp16.stat().st_size / 1e6:.0f} MB")

fp32 343 MB -> fp16 172 MB


## Does the browser model compute the same thing?

Cosine similarity between the PyTorch features and the fp16 ONNX features, without prompts and with the real
DermaMNIST VPT prompts. (int8 quantization would cut the size to 87 MB but changed about 5% of predictions in
`quant_check.py`, so the deck uses fp16.)

In [3]:
from safetensors.torch import load_file
sess = ort.InferenceSession(str(fp16), providers=["CPUExecutionProvider"])
x = np.load(ROOT / "data" / "dermamnist_224" / "test_images.npy", mmap_mode="r")[:16]
img = ((np.asarray(x).astype(np.float32) / 255 - 0.5) / 0.5).transpose(0, 3, 1, 2).copy()
P = load_file(ROOT / "weights" / "dermamnist_vpt-deep-p50_full.safetensors")["prompts"].numpy()
for name, pr in [("P = 0 (linear probe)", np.zeros((16, 12, 0, 768), np.float32)), ("P = 50 (VPT-deep)", np.broadcast_to(P, (16, *P.shape)).copy())]:
    ref = enc(torch.from_numpy(img), torch.from_numpy(pr)).numpy()
    got = sess.run(None, {"image": img, "prompts": pr})[0]
    cos = (ref * got).sum(1) / np.linalg.norm(ref, axis=1) / np.linalg.norm(got, axis=1)
    print(f"{name:22s} cosine to PyTorch: min {cos.min():.6f}")

P = 0 (linear probe)   cosine to PyTorch: min 0.999999


P = 50 (VPT-deep)      cosine to PyTorch: min 0.999997


## Where the deck gets the models

`upload_models.py` publishes `web/` to the free model repository
[valinor61/vpt-medmnist](https://huggingface.co/valinor61/vpt-medmnist). The deck fetches
`encoder.fp16.onnx`, the task's prompt file and the task's fully fine-tuned model from there on first use,
keeps them in the browser's Cache Storage, and runs them with onnxruntime-web on several threads (the site
sends cross-origin isolation headers, see `next.config.mjs`). `heads.json` holds every classification head and
the test metrics of exactly these ONNX models:

In [4]:
heads = json.loads((ROOT / "web" / "heads.json").read_text())
for t, e in heads["tasks"].items():
    print(t, {m: (v["test_torch"]["bal_acc"], v["test_web"]["bal_acc"]) for m, v in e["methods"].items()}, "(PyTorch, ONNX balanced accuracy)")
import shutil; shutil.rmtree(tmp)                     # the check files are not needed

dermamnist {'linear': (68.08, 68.52), 'vpt': (81.03, 80.97), 'full': (84.37, 84.37)} (PyTorch, ONNX balanced accuracy)
bloodmnist {'linear': (97.35, 97.37), 'vpt': (99.14, 99.14), 'full': (99.24, 99.24)} (PyTorch, ONNX balanced accuracy)
pneumoniamnist {'linear': (90.9, 90.9), 'vpt': (94.06, 94.06), 'full': (93.25, 93.25)} (PyTorch, ONNX balanced accuracy)
